## Viken KHATCHERIAN
## Formation AI Engineer
## Openclassrooms
## Projet 6 : initiez-vous au MLOps partie (1/2)
## Notebook 3 : 03_tracking_model_exp_mlflow_p6_vk.ipynb
### ETAPE 3 : Modéliser et expérimenter avec plusieurs algorithmes + tracking des expériences avec mlflow

In [23]:
# Importer les bibliothèques
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import mlflow
import mlflow.sklearn

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [24]:
# Charger les données 
X_train = pd.read_parquet("X_train_ml.parquet")

y_train = pd.read_parquet(
    "y_train_ml.parquet"
).squeeze()

print(X_train.shape)
print(y_train.shape)

print(y_train.value_counts(normalize=True))

(307511, 250)
(307511,)
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64


In [25]:
# Configuration de mlflow
mlflow.set_tracking_uri(
    "http://127.0.0.1:5000"
)

mlflow.set_experiment(
    "P06_vk_model_benchmark"
)

<Experiment: artifact_location='file:///mnt/projects/ai_engineer_projects/P06/mlruns/6', creation_time=1780669381333, experiment_id='6', last_update_time=1780669381333, lifecycle_stage='active', name='P06_vk_model_benchmark', tags={}, trace_location=None, workspace='default'>

In [26]:
# Création d'une fonction de coût métier
# On considère un Faux Négatif (FN) 10 fois plus coûteux qu'un Faux Positif (FP)

def business_cost(y_true, y_pred):

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred
    ).ravel()

    return 10 * fn + fp

In [27]:
# Validation croisée
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [28]:
# Modélisation
# Calcul d'un coefficient de pondération de la classe positive

pos_weight = (
    (y_train == 0).sum()
    /
    (y_train == 1).sum()
)

print(pos_weight)


11.387150050352467


In [30]:
models = {

    "LogisticRegression":
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            solver="saga",
            n_jobs=8,
            tol=1e-3
        ),

    "RandomForest":
        RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42,
            n_jobs=12
        ),

    "XGBoost":
        XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            scale_pos_weight=pos_weight,
            random_state=42,
            eval_metric="logloss",
            n_jobs=8,
            tree_method="hist"
        ),

    "LightGBM":
        LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            scale_pos_weight=pos_weight,
            random_state=42,
            verbose=-1,
            n_jobs=8
        ),

    "MLP":
        Pipeline([
            ("scaler", StandardScaler()),
            ("mlp", MLPClassifier(
                hidden_layer_sizes=(128, 64),
                max_iter=500,
                early_stopping=True,
                random_state=42
            ))
        ])
}

In [32]:
from sklearn.base import clone

# Evaluation des modèles et tracking mlflow
results = []

for model_name, model in models.items():

    print("=" * 60)
    print(model_name)
    print("=" * 60)

    auc_scores = []
    recall_scores = []
    precision_scores = []
    f1_scores = []
    ap_scores = []
    cost_scores = []

    for fold, (train_idx, valid_idx) in enumerate(
        cv.split(X_train, y_train)
    ):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[valid_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[valid_idx]

        # recréer un modèle propre à chaque fold
        model_fold = clone(model)

        model_fold.fit(X_tr, y_tr)

        y_prob = model_fold.predict_proba(X_val)[:, 1]
        y_pred = (y_prob >= 0.5).astype(int)

        auc_scores.append(
            roc_auc_score(y_val, y_prob)
        )

        recall_scores.append(
            recall_score(y_val, y_pred)
        )

        precision_scores.append(
            precision_score(y_val, y_pred)
        )

        f1_scores.append(
            f1_score(y_val, y_pred)
        )

        ap_scores.append(
            average_precision_score(y_val, y_prob)
        )

        cost_scores.append(
            business_cost(y_val, y_pred)
        )

    auc_mean = np.mean(auc_scores)
    auc_std = np.std(auc_scores)

    recall_mean = np.mean(recall_scores)
    precision_mean = np.mean(precision_scores)

    f1_mean = np.mean(f1_scores)

    ap_mean = np.mean(ap_scores)

    cost_mean = np.mean(cost_scores)

    print(f"AUC      : {auc_mean:.4f}")
    print(f"Recall   : {recall_mean:.4f}")
    print(f"Precision: {precision_mean:.4f}")
    print(f"F1       : {f1_mean:.4f}")
    print(f"AP       : {ap_mean:.4f}")
    print(f"Cost     : {cost_mean:.0f}")

    with mlflow.start_run(run_name=model_name):

        mlflow.set_tags({

            "project": "P06",
            "stage": "benchmark",
            "model_family": model_name,
            "solver_info": "saga for scalability",
            "mlp_early_stopping": "enabled"
        })

        mlflow.log_param("model", model_name)
        mlflow.log_param("n_features", X_train.shape[1])
        mlflow.log_param("n_samples", len(X_train))

        mlflow.log_metric("cv_auc_mean", auc_mean)
        mlflow.log_metric("cv_auc_std", auc_std)
        mlflow.log_metric("cv_recall_mean", recall_mean)
        mlflow.log_metric("cv_precision_mean", precision_mean)
        mlflow.log_metric("cv_f1_mean", f1_mean)
        mlflow.log_metric("cv_ap_mean", ap_mean)
        mlflow.log_metric("cv_business_cost", cost_mean)

        # entraînement final sur tout le dataset
        final_model = clone(model)
        final_model.fit(X_train, y_train)

        mlflow.sklearn.log_model(
            final_model,
            name="model"
        )

    results.append({

        "model": model_name,
        "auc": auc_mean,
        "recall": recall_mean,
        "precision": precision_mean,
        "f1": f1_mean,
        "ap": ap_mean,
        "cost": cost_mean
    })

LogisticRegression
AUC      : 0.5862
Recall   : 0.4962
Precision: 0.1019
F1       : 0.1690
AP       : 0.1069
Cost     : 46765


2026/06/05 17:57:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/05 17:57:55 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run LogisticRegression at: http://127.0.0.1:5000/#/experiments/6/runs/946e455291ac4372a7b3ec465b8f8cfd
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/6
RandomForest
AUC      : 0.7529
Recall   : 0.0017
Precision: 0.5014
F1       : 0.0034
AP       : 0.2200
Cost     : 49574


2026/06/05 18:00:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/05 18:00:10 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run RandomForest at: http://127.0.0.1:5000/#/experiments/6/runs/86607e7f34d3449c9e2e178d702b641f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/6
XGBoost
AUC      : 0.7775
Recall   : 0.6564
Precision: 0.1913
F1       : 0.2962
AP       : 0.2674
Cost     : 30840


...

🏃 View run XGBoost at: http://127.0.0.1:5000/#/experiments/6/runs/ef1685a972e44009b9ca3ce1e7a88eba
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/6
LightGBM
AUC      : 0.7798
Recall   : 0.6850
Precision: 0.1835
F1       : 0.2895
AP       : 0.2695
Cost     : 30769


2026/06/05 18:00:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/05 18:00:53 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run LightGBM at: http://127.0.0.1:5000/#/experiments/6/runs/0dd616bbb9f74fe397b7f55448844818
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/6
MLP
AUC      : 0.7582
Recall   : 0.0126
Precision: 0.4986
F1       : 0.0245
AP       : 0.2343
Cost     : 49085


2026/06/05 18:02:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/05 18:02:02 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run MLP at: http://127.0.0.1:5000/#/experiments/6/runs/c6e37340a8ac422a8afae39e07584850
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/6


## Analyse comparative des modèles 

### Objectif

Comparer plusieurs modèles de classification sur des métriques robustes (AUC, Recall, Precision, F1, Average Precision et coût métier), afin d’identifier un candidat pour optimisation (étape 4).



### Contexte

- **Problème** : classification déséquilibrée (~8 % de positifs)
- **Validation** : Stratified K-Fold (5 folds)

### Métriques utilisées

- ROC-AUC  
- Recall  
- Precision  
- F1-score  
- Average Precision (AP)  
- **Coût métier** :  
$$
\text{Cost} = 10 \times FN + 1 \times FP
$$

### Résultats comparatifs

| Modèle              | AUC   | Recall | Precision | F1    | AP    | Cost ↓ |
|---------------------|------|--------|-----------|-------|-------|--------|
| LogisticRegression  | 0.586 | 0.496  | 0.102     | 0.169 | 0.107 | 46765  |
| RandomForest        | 0.753 | 0.002  | 0.501     | 0.003 | 0.220 | 49574  |
| XGBoost             | 0.778 | 0.656  | 0.191     | 0.296 | 0.267 | 30840  |
| **LightGBM**            | 0.780 | **0.685**  | 0.184     | 0.290 | 0.270 | **30769**  |
| MLP                 | 0.758 | 0.013  | 0.499     | 0.025 | 0.234 | 49085  |



### Analyse des résultats

#### 1. LightGBM meilleur compromis global
- Meilleur AUC (0.780) → meilleure capacité de ranking  
- Meilleur Recall (0.685) → détecte le plus de positifs  
- Meilleur coût métier (30 769) → meilleur choix business  
- Très bon Average Precision  

C’est le modèle le plus équilibré pour un problème déséquilibré.


#### 2. XGBoost très proche du meilleur
- AUC quasi identique (0.778)  
- Recall légèrement inférieur  
- Coût légèrement plus élevé  

Très bon candidat alternatif / modèle robuste.

#### 3. Random Forest comportement problématique
- AUC correcte (0.75)  
- Recall quasi nul (0.002)  

Le modèle n’identifie quasiment pas la classe positive possiblement en raison d’un seuil implicite trop strict + déséquilibre mal géré.

#### 4. Logistic Regression
- AUC faible (0.586)  
- Recall correct mais précision très faible  

Trop simple pour la structure du problème.

#### 5. MLP
- AUC correcte (0.758)  
- Mais recall quasi nul  

Modèle instable ici (probablement scaling / convergence). Peut-être qu'un oversampling par SMOTE peut améliorer les résultats de ce modèle.

### Analyse métier

Le coût métier confirme :

- LightGBM minimise les erreurs critiques  
- Random Forest et MLP échouent à capturer les positifs  
- Logistic Regression est trop biaisée  

**Priorité métier : Recall élevé (FN très coûteux)**

### Conclusion — Étape 3

#### Modèle retenu pour l’étape 4 :
**LightGBM**

Car :
- **Meilleur coût métier** 
- Meilleur AUC global  
- Meilleur Recall  
- Bon compromis précision / rappel  
- Stable en cross-validation  

### Objectifs pour l'étape 4

**1. Optimisation des hyperparamètres**
- Optuna ou GridSearchCV  
- focus LightGBM / XGBoost  

**2. Optimisation du seuil**
- exploration 0.1 → 0.9  
- minimisation du coût métier  

**3. MLflow Model Registry**
- enregistrement du meilleur modèle après avoir identifié le meilleur run sur mlflow ui  
- versioning du "champion run" retenu (Staging vers Production)
